# Workforce Attrition Prediction — Exploratory Analysis

**Predictive modeling for employee turnover using IBM HR Analytics**

- **Source:** IBM HR Analytics Employee Attrition & Performance Dataset (Kaggle)
- **Records:** 1,470 employees
- **Target:** Attrition (Yes/No)
- **Model AUC:** 0.757 (Random Forest with balanced classes)

---

## What This Notebook Covers

1. **Attrition Overview** — Overall rate and department breakdown
2. **Demographics** — Age, gender, marital status, education
3. **Job Factors** — Overtime, job role, level, travel
4. **Satisfaction & Compensation** — Satisfaction scores + income analysis
5. **Correlation Heatmap** — Feature relationships
6. **Feature Importance** — Random Forest ranking
7. **Model Performance** — Confusion matrix + ROC curve
8. **Tenure & Career** — Years at company, role duration, promotion gaps
9. **Key Insights Summary** — Executive-ready takeaways

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Load IBM HR data
df = pd.read_csv('../data/WA_Fn-UseC_-HR-Employee-Attrition.csv')

# Basic info
print(f"Records: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Attrition rate: {df['Attrition'].value_counts()['Yes']/len(df)*100:.1f}%")
print(f"\nDepartments: {df['Department'].unique().tolist()}")

## 1. Attrition Overview

Overall attrition: **16.1%** (237 of 1,470 employees). Sales department carries the highest risk.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall attrition
attrition_counts = df['Attrition'].value_counts()
colors = ['#0d9488', '#e11d48']
axes[0].pie(attrition_counts, labels=['Stayed', 'Left'], autopct='%1.1f%%', 
            colors=colors, startangle=90, explode=[0, 0.05])
axes[0].set_title(f'Overall Attrition\n{attrition_counts["Yes"]} of {len(df)} employees', 
                fontsize=14, fontweight='bold')

# By department
dept_attrition = df.groupby('Department')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100).sort_values(ascending=True)
bars = axes[1].barh(dept_attrition.index, dept_attrition.values, 
                    color=['#0d9488', '#f59e0b', '#e11d48'], edgecolor='white', linewidth=2)
axes[1].set_xlabel('Attrition Rate (%)', fontsize=12)
axes[1].set_title('Attrition by Department', fontsize=14, fontweight='bold')
for bar, val in zip(bars, dept_attrition.values):
    axes[1].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, 
                f'{val:.1f}%', va='center', fontsize=11, fontweight='bold')

plt.suptitle('Attrition Overview', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/01_attrition_overview.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f"Overall attrition: {attrition_counts['Yes']/len(df)*100:.1f}%")
print(f"Highest risk: {dept_attrition.index[-1]} ({dept_attrition.iloc[-1]:.1f}%)")

## 2. Demographics

Age, gender, marital status, and education patterns in attrition.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Age distribution
for attrition, color in zip(['No', 'Yes'], ['#0d9488', '#e11d48']):
    subset = df[df['Attrition'] == attrition]['Age']
    axes[0, 0].hist(subset, bins=20, alpha=0.6, label=f"{'Stayed' if attrition == 'No' else 'Left'}", 
                    color=color, edgecolor='white')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Age Distribution by Attrition', fontweight='bold')
axes[0, 0].legend()

# Gender
gender_attrition = df.groupby('Gender')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100)
axes[0, 1].bar(gender_attrition.index, gender_attrition.values, 
               color=['#f59e0b', '#8b5cf6'], edgecolor='white', linewidth=2)
axes[0, 1].set_ylabel('Attrition Rate (%)')
axes[0, 1].set_title('Attrition by Gender', fontweight='bold')
for bar, val in zip(axes[0, 1].patches, gender_attrition.values):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, 
                   f'{val:.1f}%', ha='center', fontweight='bold')

# Marital status
marital_attrition = df.groupby('MaritalStatus')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100).sort_values()
axes[1, 0].barh(marital_attrition.index, marital_attrition.values, 
                color=['#0d9488', '#f59e0b', '#e11d48'], edgecolor='white')
axes[1, 0].set_xlabel('Attrition Rate (%)')
axes[1, 0].set_title('Attrition by Marital Status', fontweight='bold')
for bar, val in zip(axes[1, 0].patches, marital_attrition.values):
    axes[1, 0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, 
                   f'{val:.1f}%', va='center', fontweight='bold')

# Education
edu_order = ['Below College', 'College', 'Bachelor', 'Master', 'Doctor']
edu_attrition = df.groupby('Education')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100)
edu_attrition = edu_attrition.reindex([e for e in edu_order if e in edu_attrition.index])
axes[1, 1].bar(range(len(edu_attrition)), edu_attrition.values, 
               color='#3b82f6', edgecolor='white')
axes[1, 1].set_xticks(range(len(edu_attrition)))
axes[1, 1].set_xticklabels(edu_attrition.index, rotation=30, ha='right')
axes[1, 1].set_ylabel('Attrition Rate (%)')
axes[1, 1].set_title('Attrition by Education Level', fontweight='bold')
for bar, val in zip(axes[1, 1].patches, edu_attrition.values):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, 
                   f'{val:.1f}%', ha='center', fontweight='bold')

plt.suptitle('Attrition Demographics', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/02_attrition_demographics.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f"Age — Stayed mean: {df[df['Attrition']=='No']['Age'].mean():.1f}, Left mean: {df[df['Attrition']=='Yes']['Age'].mean():.1f}")
print(f"Gender — Male: {gender_attrition.get('Male', 0):.1f}%, Female: {gender_attrition.get('Female', 0):.1f}%")
print(f"Marital — Single highest: {marital_attrition.iloc[-1]:.1f}%")

## 3. Job Factors

Overtime is the strongest job-level predictor: 30.5% attrition with overtime vs 10.4% without.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Overtime
ot_attrition = df.groupby('OverTime')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100)
axes[0, 0].bar(ot_attrition.index, ot_attrition.values, 
               color=['#0d9488', '#e11d48'], edgecolor='white', linewidth=2)
axes[0, 0].set_ylabel('Attrition Rate (%)')
axes[0, 0].set_title('Attrition by Overtime Status', fontweight='bold')
for bar, val in zip(axes[0, 0].patches, ot_attrition.values):
    axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                   f'{val:.1f}%', ha='center', fontweight='bold', fontsize=12)

# Job role
role_attrition = df.groupby('JobRole')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100).sort_values()
axes[0, 1].barh(range(len(role_attrition)), role_attrition.values, color='#8b5cf6', edgecolor='white')
axes[0, 1].set_yticks(range(len(role_attrition)))
axes[0, 1].set_yticklabels(role_attrition.index, fontsize=8)
axes[0, 1].set_xlabel('Attrition Rate (%)')
axes[0, 1].set_title('Attrition by Job Role', fontweight='bold')

# Job level
level_attrition = df.groupby('JobLevel')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100)
axes[1, 0].bar(level_attrition.index.astype(str), level_attrition.values, 
               color='#3b82f6', edgecolor='white')
axes[1, 0].set_xlabel('Job Level')
axes[1, 0].set_ylabel('Attrition Rate (%)')
axes[1, 0].set_title('Attrition by Job Level', fontweight='bold')

# Business travel
travel_attrition = df.groupby('BusinessTravel')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100)
axes[1, 1].bar(range(len(travel_attrition)), travel_attrition.values, 
               color=['#0d9488', '#f59e0b', '#e11d48'], edgecolor='white')
axes[1, 1].set_xticks(range(len(travel_attrition)))
axes[1, 1].set_xticklabels(travel_attrition.index, rotation=20, ha='right')
axes[1, 1].set_ylabel('Attrition Rate (%)')
axes[1, 1].set_title('Attrition by Travel Frequency', fontweight='bold')

plt.suptitle('Attrition Job Factors', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/03_attrition_job_factors.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f"Overtime — Yes: {ot_attrition['Yes']:.1f}%, No: {ot_attrition['No']:.1f}%")
print(f"Highest role risk: {role_attrition.index[-1]} ({role_attrition.iloc[-1]:.1f}%)")
print(f"Travel — Frequent: {travel_attrition.iloc[-1]:.1f}%")

## 4. Satisfaction & Compensation

Lower income correlates with higher attrition risk. Satisfaction scores reveal engagement gaps.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Income distribution
for attrition, color in zip(['No', 'Yes'], ['#0d9488', '#e11d48']):
    subset = df[df['Attrition'] == attrition]['MonthlyIncome']
    axes[0, 0].hist(subset, bins=25, alpha=0.6, 
                    label=f"{'Stayed' if attrition == 'No' else 'Left'}", 
                    color=color, edgecolor='white')
axes[0, 0].set_xlabel('Monthly Income ($)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Income Distribution by Attrition', fontweight='bold')
axes[0, 0].legend()

# Income groups
df['income_group'] = pd.cut(df['MonthlyIncome'], 
                            bins=[0, 3000, 5000, 10000, float('inf')],
                            labels=['Low (<$3K)', 'Mid ($3-5K)', 'High ($5-10K)', 'Very High ($10K+)'])
income_attrition = df.groupby('income_group')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100)
axes[0, 1].bar(range(len(income_attrition)), income_attrition.values, 
               color=['#e11d48', '#f59e0b', '#0d9488', '#3b82f6'], edgecolor='white')
axes[0, 1].set_xticks(range(len(income_attrition)))
axes[0, 1].set_xticklabels(income_attrition.index, rotation=20)
axes[0, 1].set_ylabel('Attrition Rate (%)')
axes[0, 1].set_title('Attrition by Income Group', fontweight='bold')

# Job satisfaction
sat_attrition = df.groupby('JobSatisfaction')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100)
axes[1, 0].bar(sat_attrition.index, sat_attrition.values, 
               color='#8b5cf6', edgecolor='white')
axes[1, 0].set_xlabel('Job Satisfaction (1=Low, 4=High)')
axes[1, 0].set_ylabel('Attrition Rate (%)')
axes[1, 0].set_title('Attrition by Job Satisfaction', fontweight='bold')

# Environment satisfaction
env_attrition = df.groupby('EnvironmentSatisfaction')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100)
axes[1, 1].bar(env_attrition.index, env_attrition.values, 
               color='#0d9488', edgecolor='white')
axes[1, 1].set_xlabel('Environment Satisfaction (1=Low, 4=High)')
axes[1, 1].set_ylabel('Attrition Rate (%)')
axes[1, 1].set_title('Attrition by Environment Satisfaction', fontweight='bold')

plt.suptitle('Satisfaction & Compensation', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/04_attrition_satisfaction_compensation.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f"Income — Stayed mean: ${df[df['Attrition']=='No']['MonthlyIncome'].mean():,.0f}")
print(f"Income — Left mean: ${df[df['Attrition']=='Yes']['MonthlyIncome'].mean():,.0f}")
print(f"Low income attrition: {income_attrition.iloc[0]:.1f}%")

## 5. Correlation Heatmap

Feature relationships among numeric variables.

In [ ]:
# Select numeric columns
numeric_cols = ['Age', 'MonthlyIncome', 'YearsAtCompany', 'YearsInCurrentRole', 
                'TotalWorkingYears', 'NumCompaniesWorked', 'JobSatisfaction', 
                'EnvironmentSatisfaction', 'WorkLifeBalance', 'TrainingTimesLastYear',
                'PercentSalaryHike', 'PerformanceRating', 'DistanceFromHome']

# Add binary attrition
df['Attrition_Binary'] = (df['Attrition'] == 'Yes').astype(int)
corr_cols = numeric_cols + ['Attrition_Binary']

corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/05_correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# Top correlations with attrition
attrition_corr = corr['Attrition_Binary'].drop('Attrition_Binary').sort_values(key=abs, ascending=False)
print("Top correlations with attrition:")
print(attrition_corr.head(8).to_string())

## 6. Feature Importance

Random Forest feature importance ranking.

In [ ]:
# Prepare features
features = ['Age', 'MonthlyIncome', 'YearsAtCompany', 'YearsInCurrentRole',
            'TotalWorkingYears', 'NumCompaniesWorked', 'JobSatisfaction',
            'EnvironmentSatisfaction', 'WorkLifeBalance', 'OverTime',
            'BusinessTravel', 'Department', 'JobRole', 'JobLevel',
            'MaritalStatus', 'Gender', 'DistanceFromHome', 'PercentSalaryHike',
            'TrainingTimesLastYear', 'PerformanceRating']

X = df[features].copy()
y = df['Attrition_Binary']

# Encode categoricals
le_dict = {}
for col in X.select_dtypes(include='object').columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    le_dict[col] = le

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Random Forest with balanced classes
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, max_depth=12)
rf.fit(X_train, y_train)

# Feature importance
importance = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(importance)), importance.values, color='#3b82f6', edgecolor='white')
ax.set_yticks(range(len(importance)))
ax.set_yticklabels(importance.index, fontsize=10)
ax.set_xlabel('Importance Score')
ax.set_title('Random Forest Feature Importance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/06_feature_importance_rf.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("Top 5 features:")
print(importance.tail(5).to_string())

## 7. Model Performance

Confusion matrix and ROC curve. AUC = 0.757.

In [ ]:
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Stayed', 'Left'], yticklabels=['Stayed', 'Left'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix', fontweight='bold')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#3b82f6', linewidth=2.5, label=f'RF (AUC = {auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1])

plt.suptitle('Model Performance', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/07_model_performance.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f"AUC: {auc:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Stayed', 'Left']))

## 8. Tenure & Career

Years at company, role duration, and promotion patterns.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Years at company
for attrition, color in zip(['No', 'Yes'], ['#0d9488', '#e11d48']):
    subset = df[df['Attrition'] == attrition]['YearsAtCompany']
    axes[0, 0].hist(subset, bins=20, alpha=0.6, 
                    label=f"{'Stayed' if attrition == 'No' else 'Left'}", 
                    color=color, edgecolor='white')
axes[0, 0].set_xlabel('Years at Company')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Tenure Distribution', fontweight='bold')
axes[0, 0].legend()

# Years in current role
for attrition, color in zip(['No', 'Yes'], ['#0d9488', '#e11d48']):
    subset = df[df['Attrition'] == attrition]['YearsInCurrentRole']
    axes[0, 1].hist(subset, bins=15, alpha=0.6, 
                    label=f"{'Stayed' if attrition == 'No' else 'Left'}", 
                    color=color, edgecolor='white')
axes[0, 1].set_xlabel('Years in Current Role')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Role Duration', fontweight='bold')
axes[0, 1].legend()

# Promotion gap
df['promotion_gap'] = df['YearsSinceLastPromotion']
for attrition, color in zip(['No', 'Yes'], ['#0d9488', '#e11d48']):
    subset = df[df['Attrition'] == attrition]['promotion_gap']
    axes[1, 0].hist(subset, bins=15, alpha=0.6, 
                    label=f"{'Stayed' if attrition == 'No' else 'Left'}", 
                    color=color, edgecolor='white')
axes[1, 0].set_xlabel('Years Since Last Promotion')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Promotion Gap', fontweight='bold')
axes[1, 0].legend()

# Total working years
for attrition, color in zip(['No', 'Yes'], ['#0d9488', '#e11d48']):
    subset = df[df['Attrition'] == attrition]['TotalWorkingYears']
    axes[1, 1].hist(subset, bins=20, alpha=0.6, 
                    label=f"{'Stayed' if attrition == 'No' else 'Left'}", 
                    color=color, edgecolor='white')
axes[1, 1].set_xlabel('Total Working Years')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Experience Level', fontweight='bold')
axes[1, 1].legend()

plt.suptitle('Tenure & Career Patterns', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/08_attrition_tenure_career.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f"Tenure — Stayed mean: {df[df['Attrition']=='No']['YearsAtCompany'].mean():.1f} years")
print(f"Tenure — Left mean: {df[df['Attrition']=='Yes']['YearsAtCompany'].mean():.1f} years")
print(f"Promotion gap — Left: {df[df['Attrition']=='Yes']['YearsSinceLastPromotion'].mean():.1f} years")

## 9. Key Insights Summary

Executive-ready summary of all findings.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
ax.axis('off')

summary_text = f"""
WORKFORCE ATTRITION — KEY INSIGHTS
====================================

OVERALL METRICS
---------------
• Total Employees:     {len(df):,}
• Attrition Rate:      {df['Attrition'].value_counts()['Yes']/len(df)*100:.1f}% ({df['Attrition'].value_counts()['Yes']} employees)
• Model AUC:           {auc:.3f} (Random Forest, balanced classes)

HIGHEST RISK SEGMENTS
---------------------
• Department:          {dept_attrition.index[-1]} ({dept_attrition.iloc[-1]:.1f}%)
• Overtime:            {ot_attrition['Yes']:.1f}% with overtime vs {ot_attrition['No']:.1f}% without
• Income Group:        Low income (<$3K) at {income_attrition.iloc[0]:.1f}%
• Marital Status:      Single at {marital_attrition.iloc[-1]:.1f}%
• Job Role:            {role_attrition.index[-1]} ({role_attrition.iloc[-1]:.1f}%)

TOP PREDICTIVE FEATURES
-----------------------
• {importance.index[-1]} ({importance.iloc[-1]:.3f})
• {importance.index[-2]} ({importance.iloc[-2]:.3f})
• {importance.index[-3]} ({importance.iloc[-3]:.3f})
• {importance.index[-4]} ({importance.iloc[-4]:.3f})
• {importance.index[-5]} ({importance.iloc[-5]:.3f})

ACTIONABLE RECOMMENDATIONS
--------------------------
1. Overtime management — 3x attrition risk; cap overtime hours
2. Sales retention — highest department risk; targeted programs
3. Early-career support — younger employees leave more
4. Income review — low-income group at highest risk
5. Satisfaction monitoring — low satisfaction = 2x attrition
"""

ax.text(0.05, 0.95, summary_text, transform=ax.transAxes, fontsize=12,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#f8fafc', edgecolor='#e2e8f0', linewidth=2))

plt.title('Workforce Attrition — Executive Summary', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../figures/09_key_insights_summary.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---

## Summary Table

| Metric | Value | Insight |
|--------|-------|---------|
| **Overall Attrition** | 16.1% | 237 of 1,470 employees |
| **Highest Risk Dept** | Sales | 20.6% attrition rate |
| **Overtime Impact** | 3x risk | 30.5% vs 10.4% |
| **Income Correlation** | Negative | Lower = higher risk |
| **Model AUC** | 0.757 | Good predictive power |
| **Top Feature** | MonthlyIncome | Most predictive |
| **Age Pattern** | Younger leave more | Early-career risk |
| **Satisfaction** | Low = 2x risk | Engagement matters |

---

*All analysis based on 1,470 real IBM HR records. No synthetic data. No simulated records.*

**Built by:** Sierra Napier | **Source:** IBM HR Analytics (Kaggle) | **License:** MIT